# Proof of concept: perception CNN

Can a CNN read the driving state out of the camera image? Three steps, in order:

1. **Collect** rollouts on several tracks. A privileged expert drives; the camera
   and the full feature vector are recorded. Nothing is learned.
2. **Warm** one decode-free cache per resolution, so no PNG decoder runs inside
   the training loop.
3. **Search** architecture, resolution, frame stack and optimiser, ranked on
   held-out **tracks**.

No domain randomization: no DR stages at collection, `jitter=False` at load.
That is the easiest case on purpose; robustness is a later experiment.

In [1]:
import json
import os
import time

# read at import by perception.dataset, so it must be set first
os.environ.setdefault("DEEPRACER_DATA_ROOT", "data")

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch

In [2]:
from deepracer_genesis.datasets.rollout import collect_rollout_dataset
from deepracer_genesis.datasets.splits import PRINTED_TRACKS, TrackDataset
from deepracer_genesis.envs.features import PerceptionFeatures
from deepracer_genesis.experiment import CameraEnvironment
from deepracer_genesis.perception.dataset import RolloutDataset
from deepracer_genesis.perception.model import (CHANNEL_NAMES, SIGMA,
                                                PerceptionCNN, save_checkpoint)
from deepracer_genesis.randomization.spaces import Choice, FloatRange, IntRange
from torch.utils.data import DataLoader

In [3]:
SEED = 0
ROOT = "runs/proof_cnn"
DATA_ROOT = os.environ["DEEPRACER_DATA_ROOT"]

# Ranked on tracks the CNN never trains on, not on unseen frames of a track it
# has seen: near-identical frames recur across a lap, so a frame holdout mostly
# measures memorisation. Track holdout is also what train_policy_with_cnn needs.
#
# PRINTED_TRACKS have physical counterparts, so TrackDataset keeps them out of
# both train and test by construction: a CNN that has seen reinvent_base cannot
# give an honest reading of a photo of the real reinvent_base.
POOL = ("reinvent_base", "Oval_track", "Mexico_track", "China_track",
        "reInvent2019_track", "2022_reinvent_champ", "Bowtie_track",
        "New_York_Track", "caecer_loop", "2022_summit_speedway")
SPLIT = TrackDataset(names=POOL, holdout=PRINTED_TRACKS, test_fraction=0.25,
                     seed=SEED)
TRAIN_TRACKS = SPLIT.train        # 6 tracks the CNN fits on
HOLDOUT_TRACKS = SPLIT.test       # 2 unseen tracks the search ranks on
PRINTED = SPLIT.holdout           # never collected here; real-world eval only

# collection: frames per track = COLLECT_STEPS x COLLECT_ENVS
COLLECT_ENVS = 64
COLLECT_STEPS = 512
COLLECT_RESOLUTION = (160, 120)    # (w, h); the ceiling the search downscales from

# Searched resolutions as (h, w), matching RolloutDataset. A raw cache is built
# per resolution per track set: uncompressed, so 120x160 costs 57.6 KB/frame.
RESOLUTIONS = [(120, 160), (90, 120), (60, 80)]
FRAME_STACKS = [2, 4]

HPO_TRIALS = 40
EPOCHS = 6
WORKERS = 14                       # the loader bounded this before the raw cache

torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(ROOT, exist_ok=True)
print(f"device={DEVICE}  targets={list(CHANNEL_NAMES)}")
print(f"train   {TRAIN_TRACKS}")
print(f"rank on {HOLDOUT_TRACKS}")
print(f"printed {PRINTED}  (excluded entirely)")
print(f"sigma (stock per-target RMSE, the objective's normalizer)={SIGMA}")

device=cuda  targets=['lateral', 'heading', 'speed', 'yaw_rate', 'beta', 'curv@1m', 'curv@3m']
train   ('2022_reinvent_champ', 'China_track', 'Mexico_track', 'New_York_Track', 'caecer_loop', 'reInvent2019_track')
rank on ('2022_summit_speedway', 'Bowtie_track')
printed ('reinvent_base', 'Oval_track')  (excluded entirely)
sigma (stock per-target RMSE, the objective's normalizer)=(0.125, 0.064, 0.06, 0.065, 0.083, 0.122, 0.224)


## Step 1: collect

`collect_rollout_dataset` drives a privileged expert (`NoisyExpert`: track-state
steering plus Ornstein-Uhlenbeck noise, so it wanders off the centerline and
sometimes off-track) and writes `(frame, feature-vector)` rows sorted `(env, t)`.

The env must use `PerceptionFeatures` — its `cnn_target_slice` marks which
leading channels are the supervision targets, and `meta.json` records the slice
for the trainer. `frame_stack=1` here because the dataset stacks at load time.

Idempotent per track. The printed tracks are not collected at all: keeping them
out of the dataset is stronger than keeping them out of the split.

In [4]:
def collect(track: str) -> str:
    """Records one track's rollouts, or reuses them if already on disk.

    Args:
        track: Registered track name.

    Returns:
        The dataset directory.
    """
    out = os.path.join(DATA_ROOT, f"{track}_proof")
    if os.path.exists(os.path.join(out, "meta.json")):
        return out
    world = CameraEnvironment(tracks=(track,), num_envs=COLLECT_ENVS,
                              resolution=COLLECT_RESOLUTION, frame_stack=1,
                              feature_set=PerceptionFeatures)
    return collect_rollout_dataset(world, out=out, steps=COLLECT_STEPS, seed=SEED)


started = time.time()
TRAIN_SET = tuple(f"{t}_proof" for t in TRAIN_TRACKS)
HOLDOUT_SET = tuple(f"{t}_proof" for t in HOLDOUT_TRACKS)
for track in TRAIN_TRACKS + HOLDOUT_TRACKS:
    collect(track)
print(f"\ncollected in {time.time() - started:.0f}s")

# genesis/__init__.py:215 calls torch.set_default_device(device), so gs.init
# leaves the process defaulting to cuda. Every DataLoader(shuffle=True) then
# dies in RandomSampler: "Expected a 'cuda' device type for generator but
# found 'cpu'". Hand the default back before any training runs.
torch.set_default_device("cpu")
print(f"default device reset to {torch.empty(0).device}")

meta = json.load(open(os.path.join(DATA_ROOT, f"{TRAIN_TRACKS[0]}_proof",
                                   "meta.json")))
print(f"  per track     {meta['frames']:,} frames, agent {meta['agent']}")
print(f"  DR active?    appearance={bool(meta['appearance'])} "
      f"image_aug={bool(meta['image_aug'])} physics={bool(meta['physics_dr'])}")
print(f"  target slice  {meta['cnn_target_slice']}")


collected in 0s
default device reset to cpu
  per track     32,768 frames, agent NoisyExpert
  DR active?    appearance=False image_aug=False physics=False
  target slice  [0, 7]


## Step 2: warm the caches

`RolloutDataset(resolution=...)` decodes and LANCZOS-resizes every frame once
into `raw_<h>x<w>.bin`; afterwards a frame is a memmap byte slice, so no PNG
decoder runs in the training loop. Measured on one track: **1.41 ms/stack via
PNG, 0.22 ms at 120x160, 0.06 ms at 60x80**.

Building here rather than inside a trial keeps the one-time cost out of the
search. Note the default-device reset at the end of step 1: collection
initialises genesis, which leaves torch defaulting to CUDA, and every shuffled
DataLoader fails until it is handed back. The raw caches are uncompressed — budget roughly
`frames x h x w x 3` bytes per resolution per track set.

In [5]:
DATASETS = {}


def dataset_for(tracks: tuple, k: int, resolution: tuple) -> RolloutDataset:
    """Returns a memoised dataset, building its raw cache on first use.

    Args:
        tracks: Dataset directory names.
        k: Frames per stack.
        resolution: ``(h, w)`` served to the model.

    Returns:
        The dataset.
    """
    key = (tracks, k, resolution)
    if key not in DATASETS:
        DATASETS[key] = RolloutDataset(tracks=tracks, k=k, jitter=False,
                                       seed=SEED, resolution=resolution)
    return DATASETS[key]


rows = []
for resolution in RESOLUTIONS:
    for k in FRAME_STACKS:
        for name, tracks in (("train", TRAIN_SET), ("holdout", HOLDOUT_SET)):
            started = time.time()
            ds = dataset_for(tracks, k, resolution)
            rows.append({"split": name, "k": k, "stacks": len(ds),
                         "resolution": f"{resolution[0]}x{resolution[1]}",
                         "build_s": round(time.time() - started, 1)})
display(pd.DataFrame(rows))

,split,k,stacks,resolution,build_s
0,train,2,195373,120x160,0.0
1,holdout,2,65100,120x160,0.0
2,train,4,192914,120x160,0.0
3,holdout,4,64230,120x160,0.0
4,train,2,195373,90x120,0.0
5,holdout,2,65100,90x120,0.0
6,train,4,192914,90x120,0.0
7,holdout,4,64230,90x120,0.0
8,train,2,195373,60x80,0.0
9,holdout,2,65100,60x80,0.0


## Step 3: search

Ranked on **sigma-normalized** MSE: each target is divided by the stock model's
recorded RMSE (`model.SIGMA`), so channels with different natural scales
contribute comparably and the score is a multiple of "as good as the stock net".
Below 1.0 beats it.

A trunk that collapses the spatial dimensions raises from `PerceptionCNN`; those
draws are pruned before any training, with the reason recorded.

In [6]:
def suggest_arch(trial: optuna.Trial) -> dict:
    """Samples the convolution trunk and head width.

    Depth is drawn once and the three tuples derive from it, so a length
    mismatch is unrepresentable.

    Args:
        trial: The Optuna trial.

    Returns:
        Keyword arguments for ``PerceptionCNN``.
    """
    depth = trial.suggest_int("depth", 3, 5)
    base = trial.suggest_categorical("base_channels", [16, 32, 48])
    k0 = trial.suggest_categorical("k0", [7, 5, 3])
    s0 = trial.suggest_categorical("s0", [4, 2])
    return {
        "channels": tuple(min(base * 2 ** min(i, 2), 128) for i in range(depth)),
        "kernels": (k0,) + (3,) * (depth - 1),
        "strides": (s0,) + (2,) * (depth - 1),
        "head": trial.suggest_categorical("head", [64, 128, 256]),
    }


def normalized_mse(per_target: np.ndarray) -> float:
    """Scales each target's MSE by the stock model's variance for that channel.

    Args:
        per_target: Per-target mean squared error.

    Returns:
        The mean across targets; 1.0 matches the stock network.
    """
    return float(np.mean(per_target / np.square(SIGMA[:len(per_target)])))


def run_epoch(model, loader, optimizer=None) -> np.ndarray:
    """One pass over a loader, returning per-target mean squared error.

    Args:
        model: The network.
        loader: Yields (frames, target).
        optimizer: Set to train; None to evaluate under no_grad.

    Returns:
        Per-target MSE.
    """
    training = optimizer is not None
    model.train(training)
    total, seen = np.zeros(len(CHANNEL_NAMES)), 0
    with torch.set_grad_enabled(training):
        for frames, target in loader:
            frames = frames.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)
            per = (model(frames) - target).pow(2).mean(dim=0)
            if training:
                optimizer.zero_grad()
                per.mean().backward()
                optimizer.step()
            total += per.detach().cpu().numpy() * frames.shape[0]
            seen += frames.shape[0]
    return total / seen

In [7]:
def objective(trial: optuna.Trial) -> float:
    """Trains one architecture and scores it on held-out tracks.

    Args:
        trial: The Optuna trial.

    Returns:
        Sigma-normalized validation MSE; below 1.0 beats the stock network.

    Raises:
        optuna.TrialPruned: On an unbuildable trunk or a Hyperband cut.
    """
    resolution = RESOLUTIONS[trial.suggest_int("res_idx", 0, len(RESOLUTIONS) - 1)]
    k = trial.suggest_categorical("k", FRAME_STACKS)
    arch = suggest_arch(trial)
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    batch = trial.suggest_categorical("batch", [128, 256, 512])
    cfg = {"resolution": resolution, "k": k, "lr": lr, "batch": batch,
           "weight_decay": weight_decay, **arch}
    trial.set_user_attr("cfg", {**cfg, "channels": list(arch["channels"]),
                                "kernels": list(arch["kernels"]),
                                "strides": list(arch["strides"])})

    try:
        model = PerceptionCNN(in_channels=3 * k, n_targets=len(CHANNEL_NAMES),
                              input_hw=resolution, **arch).to(DEVICE)
    except ValueError as exc:
        trial.set_user_attr("outcome", "unbuildable")
        trial.set_user_attr("reason", str(exc)[:120])
        raise optuna.TrialPruned(str(exc)[:120])

    train_dl = DataLoader(dataset_for(TRAIN_SET, k, resolution), batch_size=batch,
                          shuffle=True, num_workers=WORKERS, pin_memory=True,
                          persistent_workers=True, drop_last=True)
    val_dl = DataLoader(dataset_for(HOLDOUT_SET, k, resolution), batch_size=batch,
                        num_workers=4, pin_memory=True, persistent_workers=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr,
                                 weight_decay=weight_decay)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best = float("inf")
    try:
        for epoch in range(EPOCHS):
            run_epoch(model, train_dl, optimizer)
            score = normalized_mse(run_epoch(model, val_dl))
            schedule.step()
            best = min(best, score)
            trial.report(score, epoch)
            if trial.should_prune():
                trial.set_user_attr("outcome", "hyperband")
                trial.set_user_attr("reason",
                                    f"score {score:.3f} at epoch {epoch}")
                raise optuna.TrialPruned()
    finally:
        # persistent_workers keeps 18 processes alive per trial otherwise
        del train_dl, val_dl

    trial.set_user_attr("outcome", "completed")
    trial.set_user_attr("reason", "")
    trial.set_user_attr("params", sum(p.numel() for p in model.parameters()))
    save_checkpoint(model, os.path.join(ROOT, f"t{trial.number:03d}.pt"))
    return best


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(
    direction="minimize", study_name="proof_cnn",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.HyperbandPruner(min_resource=1, max_resource=EPOCHS,
                                          reduction_factor=3),
    storage=f"sqlite:///{ROOT}/study.db", load_if_exists=True)
# catch: one bad trial marks FAIL and the study continues instead of dying
study.optimize(objective, n_trials=HPO_TRIALS, catch=(RuntimeError,))

/home/lunav0/.SecretProjects/deepracer-studies/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 14 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/lunav0/.SecretProjects/deepracer-studies/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 14 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessar

## Result

`score` is sigma-normalized MSE on held-out tracks: **below 1.0 beats the stock
network**. The per-target breakdown is the interesting part — near-field
geometry (lateral, heading) should be readable from one frame, far-field
curvature often is not.

In [8]:
table = pd.DataFrame([
    {"trial": t.number, "state": t.state.name,
     "outcome": t.user_attrs.get("outcome", "unrecorded"),
     "reason": t.user_attrs.get("reason", ""),
     "score": t.value, "params": t.user_attrs.get("params"), **t.params}
    for t in study.trials])
display(table.sort_values("score").head(15))
print("outcomes:", table["outcome"].value_counts().to_dict())

BEST = study.best_trial.user_attrs["cfg"]
print(f"\nwinner: score {study.best_value:.3f}  {BEST}")
print(f"checkpoint: {ROOT}/t{study.best_trial.number:03d}.pt  "
      "(architecture travels with it)")

,trial,state,outcome,reason,score,params,res_idx,k,depth,base_channels,k0,s0,head,lr,weight_decay,batch
38,38,COMPLETE,completed,,1.336337,996967.0,0,4,3,32,7,4,128,0.000813,0.000004,512
34,34,COMPLETE,completed,,1.370393,996967.0,0,4,3,32,7,4,128,0.000678,0.000005,512
20,20,COMPLETE,completed,,1.382044,623447.0,0,4,3,48,7,4,64,0.001113,0.000002,512
18,18,COMPLETE,completed,,1.386114,623447.0,0,4,3,48,7,4,64,0.001059,0.000021,512
32,32,COMPLETE,completed,,1.394752,505447.0,1,4,3,32,7,4,128,0.000700,0.000005,512
17,17,COMPLETE,completed,,1.413724,623447.0,0,4,3,48,7,4,64,0.000507,0.000028,512
31,31,COMPLETE,completed,,1.438896,505447.0,1,4,3,32,7,4,128,0.000641,0.000006,512
15,15,COMPLETE,completed,,1.461238,623447.0,0,4,3,48,7,4,64,0.000290,0.000017,512
23,23,PRUNED,hyperband,score 1.501 at epoch 3,1.501135,NaN,0,4,3,48,7,4,64,0.000533,0.000004,512
33,33,PRUNED,hyperband,score 1.502 at epoch 3,1.502419,NaN,1,4,3,32,7,4,128,0.000965,0.000002,512


outcomes: {'hyperband': 22, 'completed': 13, 'unbuildable': 5, 'unrecorded': 1}

winner: score 1.336  {'resolution': [120, 160], 'k': 4, 'lr': 0.0008130165503706205, 'batch': 512, 'weight_decay': 3.542085596025971e-06, 'channels': [32, 64, 128], 'kernels': [7, 3, 3], 'strides': [4, 2, 2], 'head': 128}
checkpoint: runs/proof_cnn/t038.pt  (architecture travels with it)


## Notes

- Every completed trial saves a checkpoint through `save_checkpoint`, so the
  architecture travels with the weights. `CNNPerceptionFeatures` and
  `export_perception_cnn` both rebuild from it, so a winner here drops straight
  into the sim or onto the car.
- Ranking is on held-out tracks. A frame holdout on one track would mostly
  measure memorisation, since near-identical frames recur every lap.
- `reinvent_base` and `Oval_track` are never collected, so nothing here has
  seen the geometry you own physically. That is what keeps a sim2real
  comparison on those tracks honest.
- `jitter=True` on the training dataset is the natural next experiment: it
  measures what camera augmentation costs, and it composes with everything here.